# Auric CV assignment: Colab runner
Thin driver around `src/`. Drive layout: `MyDrive/auric/{CV_Assignment_AAL.zip, code/, runs/, tiles.zip}`.
After a disconnect: re-run all cells; `train.py` resumes from `last.pt`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
D = '/content/drive/MyDrive/auric'
%cd /content/drive/MyDrive/auric/code
!pip -q install -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# data: raw images to local disk (Drive reads are slow); tiles cached on Drive after the first session
import os
if not os.path.exists('/content/CV_Assignment_AAL'):
    !unzip -q $D/CV_Assignment_AAL.zip -x '__MACOSX/*' -d /content
if not os.path.exists('/content/tiles'):
    if os.path.exists(f'{D}/tiles.zip'):
        !unzip -q $D/tiles.zip -d /content
    else:
        !python src/tile.py --root /content/CV_Assignment_AAL --out /content/tiles
        !cd /content && zip -q -0 -r $D/tiles.zip tiles

In [ ]:
EXP = 'E0'  # config name in configs/
!mkdir -p $D/runs && pip freeze > $D/runs/pip_freeze_{EXP}.txt
!python src/train.py --config configs/{EXP}.yaml --tiles /content/tiles --project $D/runs

In [ ]:
# evaluate: sliced inference on full val images -> evaluator (single source of truth)
import yaml
RUN = yaml.safe_load(open(f'configs/{EXP}.yaml'))['name']
!python src/predict.py --weights $D/runs/{RUN}/weights/last.pt --split val --root /content/CV_Assignment_AAL --out experiments/results/{RUN}_val_preds.csv
!python src/evaluate.py --preds experiments/results/{RUN}_val_preds.csv --name {RUN} --root /content/CV_Assignment_AAL